# 08 — Impact Simulation


In [ ]:
import itertools
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch

from build_optimiser.config import Config
from build_optimiser.graph import attach_metrics, load_graph
from build_optimiser.simulation import (
    feature_subset_build_time,
    replay_git_history,
    sensitivity_analysis,
    simulate_incremental_build,
)

cfg = Config.from_yaml('../config.yaml')
file_df = pd.read_parquet('../data/processed/file_metrics.parquet')
target_df = pd.read_parquet('../data/processed/target_metrics.parquet')
edge_df = pd.read_parquet('../data/processed/edge_list.parquet')
G = load_graph('../data/processed/edge_list.parquet')
attach_metrics(G, target_df)

%matplotlib inline
sns.set_theme(style='whitegrid')

## 1. Load Data

Load feature group assignments, contributor groups, and target ownership from NB05/NB06 outputs.


In [ ]:
# ── Feature group assignments (NB06) ─────────────────────────────────────────
fg_df = pd.read_parquet('../data/processed/feature_group_assignments.parquet')
group_map = dict(zip(fg_df['cmake_target'], fg_df['feature_group']))
all_groups = sorted(fg_df['feature_group'].unique())
feature_groups_only = [g for g in all_groups if g != 'core']
core_targets = set(fg_df[fg_df['feature_group'] == 'core']['cmake_target'])

# ── Contributor groups (NB contributor analysis) ──────────────────────────────
contrib_groups_path = Path('../data/processed/contributor_groups.parquet')
if contrib_groups_path.exists():
    contrib_groups_df = pd.read_parquet(contrib_groups_path)
else:
    contrib_groups_df = pd.DataFrame(columns=['contributor', 'group_id', 'group_label', 'group_score'])
    print('Warning: contributor_groups.parquet not found.')

# ── Target ownership (contributor analysis) ───────────────────────────────────
ownership_path = Path('../data/processed/target_ownership.parquet')
if ownership_path.exists():
    ownership_df = pd.read_parquet(ownership_path)
    # Derive owning_group_label if not present
    if 'owning_group_label' not in ownership_df.columns and 'owning_group_id' in ownership_df.columns and contrib_groups_path.exists():
        label_map = contrib_groups_df[['group_id', 'group_label']].drop_duplicates().set_index('group_id')['group_label']
        ownership_df['owning_group_label'] = ownership_df['owning_group_id'].map(label_map).fillna('unknown')
    bus_factor_col = 'bus_factor' if 'bus_factor' in ownership_df.columns else None
    bus_factor_map = dict(zip(ownership_df['cmake_target'], ownership_df[bus_factor_col])) if bus_factor_col else {}
    label_col = 'owning_group_label' if 'owning_group_label' in ownership_df.columns else 'owning_group_id'
    ownership_map = dict(zip(ownership_df['cmake_target'], ownership_df[label_col]))
else:
    ownership_df = pd.DataFrame()
    bus_factor_map = {}
    ownership_map = {}
    print('Warning: target_ownership.parquet not found.')

# ── Git history detail ────────────────────────────────────────────────────────
git_history_path = Path('../data/raw/git_history_detail.csv')
if git_history_path.exists():
    git_detail_df = pd.read_csv(git_history_path)
    if 'commit_date' in git_detail_df.columns:
        git_detail_df['commit_date'] = pd.to_datetime(git_detail_df['commit_date'], utc=True)
else:
    git_detail_df = pd.DataFrame(columns=['commit_hash', 'author_email', 'source_file', 'commit_date'])
    print('Warning: git_history_detail.csv not found.')

# ── Exe-library matrix (NB05) ─────────────────────────────────────────────────
exe_lib_path = Path('../data/processed/exe_library_matrix.parquet')
if exe_lib_path.exists():
    exe_lib_df = pd.read_parquet(exe_lib_path)
else:
    from build_optimiser.features import compute_exe_library_matrix
    exe_lib_df = compute_exe_library_matrix(G, target_df[['cmake_target', 'target_type']])

# ── Build target_times dict ───────────────────────────────────────────────────
target_times = {
    row['cmake_target']: float(row.get('total_build_time_ms') or 0)
    for _, row in target_df.iterrows()
    if row['cmake_target'] in G
}
N_CORES = getattr(cfg, 'n_cores', 8)

# ── File → target mapping ──────────────────────────────────────────────────────
file_to_target = dict(zip(file_df['source_file'].dropna(), file_df['cmake_target'].dropna()))

print(f'Targets:           {G.number_of_nodes():>6}')
print(f'Feature groups:    {len(all_groups):>6}')
print(f'Contributors:      {len(contrib_groups_df):>6}')
print(f'Git detail rows:   {len(git_detail_df):>6}')
print(f'N cores for sim:   {N_CORES:>6}')

## 2. Per-Team Build Time Simulation

Replay git history with `replay_git_history()`. For each team: current (all targets) vs proposed (team's feature groups + core + transitive dependencies). Report mean / P90 build time and reduction.


In [ ]:
# ── Map contributor → team (group_label) ──────────────────────────────────────
contributor_team: dict[str, str] = {}
if not contrib_groups_df.empty:
    contrib_col = 'group_label' if 'group_label' in contrib_groups_df.columns else 'group_id'
    contributor_team = dict(zip(contrib_groups_df['contributor'].astype(str),
                                 contrib_groups_df[contrib_col].astype(str)))

# ── Enrich git history with team labels ───────────────────────────────────────
if not git_detail_df.empty and contributor_team:
    auth_col = 'author_email' if 'author_email' in git_detail_df.columns else 'contributor'
    git_detail_df['team'] = git_detail_df[auth_col].map(contributor_team).fillna('unknown')
else:
    git_detail_df['team'] = 'unknown'

teams = [t for t in git_detail_df['team'].unique() if t != 'unknown']
print(f'Teams identified: {len(teams)}')

# ── Build enabled_targets_per_team (proposed: team's feature groups + core + deps) ──
# Team → set of feature groups they own targets in
team_groups: dict[str, set[str]] = {}
if not ownership_df.empty and not contrib_groups_df.empty:
    owner_col = 'owning_group_label' if 'owning_group_label' in ownership_df.columns else 'owning_group_id'
    for _, row in ownership_df.iterrows():
        team_lbl = str(row[owner_col])
        fg = group_map.get(row['cmake_target'], 'core')
        if team_lbl not in team_groups:
            team_groups[team_lbl] = set()
        team_groups[team_lbl].add(fg)

enabled_targets_per_team: dict[str, set[str]] = {}
for team, grps in team_groups.items():
    enabled_grps = grps | {'core'}
    enabled = {t for t in G.nodes() if group_map.get(t, 'core') in enabled_grps}
    # Add transitive dependencies of enabled targets
    all_deps = set()
    for t in enabled:
        all_deps |= nx.descendants(G, t)
    enabled_targets_per_team[team] = enabled | all_deps

print(f'Teams with enabled target sets: {len(enabled_targets_per_team)}')
for team, enabled in enabled_targets_per_team.items():
    print(f'  {team}: {len(enabled)} enabled targets')

# ── Run replay simulations ────────────────────────────────────────────────────
if not git_detail_df.empty and 'commit_hash' in git_detail_df.columns:
    # Align source_file column
    src_col = 'source_file' if 'source_file' in git_detail_df.columns else 'file_path'
    if src_col not in git_detail_df.columns:
        print('Warning: no source_file column in git history — skipping replay.')
        replay_all = pd.DataFrame()
        replay_restricted = pd.DataFrame()
    else:
        replay_cols = git_detail_df.rename(columns={src_col: 'source_file'})

        # Current (all targets)
        replay_all = replay_git_history(
            G, replay_cols, file_to_target, target_times, N_CORES,
            enabled_targets_per_team=None,
        )
        replay_all['scenario'] = 'current'

        # Proposed (team-restricted targets)
        replay_restricted = replay_git_history(
            G, replay_cols, file_to_target, target_times, N_CORES,
            enabled_targets_per_team=enabled_targets_per_team if enabled_targets_per_team else None,
        )
        replay_restricted['scenario'] = 'proposed'
else:
    replay_all = pd.DataFrame()
    replay_restricted = pd.DataFrame()
    print('Skipping replay: no git history data available.')

# ── Compare per-team build times ──────────────────────────────────────────────
if not replay_all.empty and not replay_restricted.empty:
    combined_replay = pd.concat([replay_all, replay_restricted], ignore_index=True)

    team_stats = []
    for team in combined_replay['team'].unique():
        for scenario in ['current', 'proposed']:
            sub = combined_replay[
                (combined_replay['team'] == team) &
                (combined_replay['scenario'] == scenario) &
                (combined_replay['build_time_ms'] > 0)
            ]
            if sub.empty:
                continue
            team_stats.append({
                'team': team,
                'scenario': scenario,
                'n_commits': len(sub),
                'mean_ms': sub['build_time_ms'].mean(),
                'p90_ms': sub['build_time_ms'].quantile(0.90),
                'p50_ms': sub['build_time_ms'].quantile(0.50),
            })

    team_stats_df = pd.DataFrame(team_stats)

    # Pivot for comparison
    if not team_stats_df.empty:
        pivot = team_stats_df.pivot_table(
            index='team',
            columns='scenario',
            values=['mean_ms', 'p90_ms'],
        ).reset_index()
        pivot.columns = ['_'.join(c).strip('_') for c in pivot.columns]
        if 'mean_ms_current' in pivot.columns and 'mean_ms_proposed' in pivot.columns:
            pivot['mean_reduction_pct'] = (
                (pivot['mean_ms_current'] - pivot['mean_ms_proposed']) /
                pivot['mean_ms_current'].replace(0, np.nan) * 100
            ).fillna(0)
        if 'p90_ms_current' in pivot.columns and 'p90_ms_proposed' in pivot.columns:
            pivot['p90_reduction_pct'] = (
                (pivot['p90_ms_current'] - pivot['p90_ms_proposed']) /
                pivot['p90_ms_current'].replace(0, np.nan) * 100
            ).fillna(0)

        print('Per-team build time comparison:')
        display(pivot.sort_values('mean_reduction_pct', ascending=False).reset_index(drop=True))

        # ── Grouped bar chart ─────────────────────────────────────────────────
        plot_teams = [t for t in team_stats_df['team'].unique() if t != 'unknown'][:10]
        if plot_teams:
            fig, axes = plt.subplots(1, 2, figsize=(16, 5))
            for ax, metric, label in [
                (axes[0], 'mean_ms', 'Mean build time (ms)'),
                (axes[1], 'p90_ms', 'P90 build time (ms)'),
            ]:
                x = np.arange(len(plot_teams))
                width = 0.35
                for i, (scenario, color) in enumerate([('current', 'steelblue'), ('proposed', '#55A868')]):
                    vals = [
                        team_stats_df[
                            (team_stats_df['team'] == t) & (team_stats_df['scenario'] == scenario)
                        ][metric].values[0]
                        if len(team_stats_df[
                            (team_stats_df['team'] == t) & (team_stats_df['scenario'] == scenario)
                        ]) > 0 else 0
                        for t in plot_teams
                    ]
                    ax.bar(x + i * width, vals, width, label=scenario.capitalize(), color=color, alpha=0.8)
                ax.set_xticks(x + width / 2)
                ax.set_xticklabels([t[:15] for t in plot_teams], rotation=30, ha='right', fontsize=8)
                ax.set_ylabel(label)
                ax.set_title(f'Per-Team {label}: Current vs Proposed')
                ax.legend(fontsize=8)
            plt.tight_layout()
            plt.show()
else:
    print('No replay results available to compare.')

## 3. Full Build Time Under Feature Subsets

Use `feature_subset_build_time()` for combinations: core only, core + each group, core + each pair, all groups. Present as table and bar chart.


In [ ]:
# ── Enumerate combinations to evaluate ───────────────────────────────────────
combinations_to_eval = []

# Baseline: core only
combinations_to_eval.append(('core only', []))

# Core + each individual group
for grp in feature_groups_only:
    combinations_to_eval.append((f'core + {grp}', [grp]))

# Core + each pair (capped at 10 pairs to avoid combinatorial explosion)
pairs = list(itertools.combinations(feature_groups_only, 2))
for g1, g2 in pairs[:10]:
    combinations_to_eval.append((f'core + {g1} + {g2}', [g1, g2]))

# All groups
combinations_to_eval.append(('all groups', feature_groups_only))

print(f'Evaluating {len(combinations_to_eval)} combinations...')

# ── Compute build time for each combination ───────────────────────────────────
subset_rows = []
for label, group_combo in combinations_to_eval:
    bt = feature_subset_build_time(G, fg_df, group_combo, target_times, N_CORES)
    enabled_grps = set(group_combo) | {'core'}
    enabled_count = sum(
        1 for t in G.nodes() if group_map.get(t, 'core') in enabled_grps
    )
    subset_rows.append({
        'combination': label,
        'groups_enabled': len(enabled_grps),
        'targets_enabled': enabled_count,
        'build_time_ms': bt,
        'build_time_s': bt / 1000,
    })

subset_df = pd.DataFrame(subset_rows)

# Compute savings vs all groups
full_build_ms = subset_df[subset_df['combination'] == 'all groups']['build_time_ms'].values[0]
subset_df['saving_vs_full_pct'] = (
    (full_build_ms - subset_df['build_time_ms']) / max(full_build_ms, 1) * 100
).round(1)

print(f'Full build time (all groups): {full_build_ms:,.0f} ms ({full_build_ms / 1000:.1f} s)')
display(subset_df[['combination', 'targets_enabled', 'build_time_s',
                    'saving_vs_full_pct']].to_string(index=False))

# ── Bar chart ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, max(5, len(subset_df) * 0.4)))
colors = sns.color_palette('Blues_d', n_colors=len(subset_df))
ax.barh(range(len(subset_df)), subset_df['build_time_s'].values, color=colors)
ax.set_yticks(range(len(subset_df)))
ax.set_yticklabels(subset_df['combination'].values, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Simulated full build time (seconds)')
ax.set_title('Build Time by Feature Group Combination')
# Annotate with saving
for i, row in enumerate(subset_df.itertuples()):
    if row.saving_vs_full_pct > 0:
        ax.text(row.build_time_s + 0.5, i, f'-{row.saving_vs_full_pct:.0f}%',
                va='center', fontsize=7, color='darkgreen')
plt.tight_layout()
plt.show()

## 4. Sensitivity Analysis on K

Use `sensitivity_analysis()` with K from 2-10. Plot cross_group_edges and core_size vs K. Identify the elbow point.


In [ ]:
# ── Run sensitivity analysis ──────────────────────────────────────────────────
k_range = range(2, 11)
print(f'Running sensitivity analysis for K = {list(k_range)}...')

sens_df = sensitivity_analysis(G, exe_lib_df, target_times, k_range, N_CORES)

print('Sensitivity analysis results:')
display(sens_df.reset_index(drop=True))

# ── Elbow detection ───────────────────────────────────────────────────────────
if len(sens_df) >= 3:
    cross_vals = sens_df['cross_group_edges'].values
    k_vals = sens_df['k'].values
    # Second difference to detect elbow (minimum curvature change)
    if len(cross_vals) >= 3:
        diffs = np.diff(cross_vals)
        second_diffs = np.diff(diffs)
        elbow_idx = np.argmax(second_diffs) + 1  # +1 offset for second diff
        elbow_k = k_vals[elbow_idx] if elbow_idx < len(k_vals) else k_vals[-1]
    else:
        elbow_k = k_vals[len(k_vals) // 2]
    print(f'\nElbow detected at K = {elbow_k}')
else:
    elbow_k = None
    print('Not enough data points for elbow detection.')

# ── Plot ──────────────────────────────────────────────────────────────────────
if not sens_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.plot(sens_df['k'], sens_df['cross_group_edges'], 'o-', color='steelblue', linewidth=2)
    if elbow_k is not None:
        ax.axvline(elbow_k, color='red', linestyle='--', alpha=0.7, label=f'Elbow K={elbow_k}')
        ax.legend(fontsize=9)
    ax.set_xlabel('Number of feature groups (K)')
    ax.set_ylabel('Cross-group edges')
    ax.set_title('Cross-Group Edges vs K')
    ax.set_xticks(sens_df['k'].values)

    ax = axes[1]
    ax.plot(sens_df['k'], sens_df['core_size'], 's-', color='darkorange', linewidth=2, label='Core size')
    if 'actual_groups' in sens_df.columns:
        ax2 = ax.twinx()
        ax2.plot(sens_df['k'], sens_df['actual_groups'], 'D--', color='#55A868',
                 linewidth=1.5, alpha=0.8, label='Actual groups')
        ax2.set_ylabel('Actual number of groups', color='#55A868')
        ax2.tick_params(axis='y', labelcolor='#55A868')
    if elbow_k is not None:
        ax.axvline(elbow_k, color='red', linestyle='--', alpha=0.7)
    ax.set_xlabel('Number of feature groups (K)')
    ax.set_ylabel('Core size (targets)')
    ax.set_title('Core Size vs K')
    ax.set_xticks(sens_df['k'].values)
    ax.legend(loc='upper left', fontsize=9)

    plt.tight_layout()
    plt.show()

## 5. Schema Change Impact

For each codegen target: compute change frequency × total downstream compile time. Compare under full build vs feature group restriction to report reduction.


In [ ]:
# ── Identify codegen targets ───────────────────────────────────────────────────
codegen_targets = target_df[target_df['codegen_file_count'].fillna(0) > 0].copy()
working_days = cfg.git_history_months * 20
rev_G = G.reverse()

schema_rows = []
for _, row in codegen_targets.iterrows():
    t = row['cmake_target']
    if t not in G:
        continue

    # Change frequency
    commit_count = int(row.get('git_commit_count_total') or 0)
    change_freq = commit_count / working_days if working_days > 0 else 0.0

    # Downstream targets in full build
    downstream_all = nx.descendants(rev_G, t) if t in rev_G else set()
    downstream_compile_ms = sum(
        float(G.nodes[n].get('compile_time_sum_ms') or 0)
        for n in downstream_all if n in G
    )
    full_impact = change_freq * downstream_compile_ms

    # Downstream in restricted build (only targets in same or core group)
    codegen_grp = group_map.get(t, 'core')
    allowed_groups = {codegen_grp, 'core'}
    downstream_restricted = {
        n for n in downstream_all
        if group_map.get(n, 'core') in allowed_groups
    }
    restricted_compile_ms = sum(
        float(G.nodes[n].get('compile_time_sum_ms') or 0)
        for n in downstream_restricted if n in G
    )
    restricted_impact = change_freq * restricted_compile_ms

    schema_rows.append({
        'cmake_target': t,
        'feature_group': codegen_grp,
        'codegen_file_count': int(row.get('codegen_file_count') or 0),
        'commit_count': commit_count,
        'change_freq': round(change_freq, 4),
        'downstream_targets_full': len(downstream_all),
        'downstream_targets_restricted': len(downstream_restricted),
        'full_impact_ms_per_day': round(full_impact, 1),
        'restricted_impact_ms_per_day': round(restricted_impact, 1),
        'impact_reduction_pct': round(
            (full_impact - restricted_impact) / max(full_impact, 1) * 100, 1
        ),
    })

schema_df = pd.DataFrame(schema_rows).sort_values('full_impact_ms_per_day', ascending=False)

print(f'Codegen targets analysed: {len(schema_df)}')
total_full = schema_df['full_impact_ms_per_day'].sum()
total_restricted = schema_df['restricted_impact_ms_per_day'].sum()
print(f'Total schema change impact (full build):       {total_full:,.0f} ms/day')
print(f'Total schema change impact (feature-scoped):  {total_restricted:,.0f} ms/day')
print(f'Reduction: {(total_full - total_restricted) / max(total_full, 1):.1%}')
print()
display(schema_df[['cmake_target', 'feature_group', 'codegen_file_count',
                    'downstream_targets_full', 'downstream_targets_restricted',
                    'full_impact_ms_per_day', 'restricted_impact_ms_per_day',
                    'impact_reduction_pct']].head(20).reset_index(drop=True))

# ── Dual bar chart: full vs restricted impact ─────────────────────────────────
top_schema = schema_df[schema_df['full_impact_ms_per_day'] > 0].head(12)
if not top_schema.empty:
    fig, ax = plt.subplots(figsize=(14, max(4, len(top_schema) * 0.5)))
    x = np.arange(len(top_schema))
    width = 0.35
    ax.barh(x + width / 2, top_schema['full_impact_ms_per_day'].values,
            width, label='Full build', color='#C44E52', alpha=0.8)
    ax.barh(x - width / 2, top_schema['restricted_impact_ms_per_day'].values,
            width, label='Feature-scoped', color='#55A868', alpha=0.8)
    ax.set_yticks(x)
    ax.set_yticklabels(top_schema['cmake_target'].apply(lambda s: s[:30]).values, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Impact (ms / working day)')
    ax.set_title('Schema Change Impact: Full vs Feature-Scoped Build')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

## 6. Risk Analysis

Single-person bus factor on critical targets. Core dependency depth. Target splits with high internal coupling. Rate each risk.


In [ ]:
# ── Critical path ─────────────────────────────────────────────────────────────
weight_attr = 'effective_weight'
for n in G.nodes():
    nd = G.nodes[n]
    G.nodes[n][weight_attr] = (
        (nd.get('codegen_time_ms') or 0)
        + (nd.get('compile_time_max_ms') or 0)
        + (nd.get('archive_time_ms') or 0)
        + (nd.get('link_time_ms') or 0)
    )

try:
    topo = list(nx.topological_sort(G))
    earliest_finish = {}
    for n in reversed(topo):
        w = G.nodes[n][weight_attr]
        deps = list(G.successors(n))
        earliest_finish[n] = max((earliest_finish[d] for d in deps), default=0) + w
    cp_end = max(earliest_finish, key=earliest_finish.get)
    cp_length = earliest_finish[cp_end]
    # Reconstruct critical path
    cp_nodes = []
    node = cp_end
    predecessors_on_path = {}
    for n in reversed(topo):
        deps = list(G.successors(n))
        if deps:
            predecessors_on_path[n] = max(deps, key=lambda d: earliest_finish[d])
        else:
            predecessors_on_path[n] = None
    while node is not None:
        cp_nodes.append(node)
        node = predecessors_on_path.get(node)
    cp_set = set(cp_nodes)
    print(f'Critical path length: {cp_length:,.0f} ms ({len(cp_nodes)} nodes)')
except nx.NetworkXUnfeasible:
    cp_set = set()
    cp_length = 0
    print('Warning: graph has cycles — cannot compute critical path.')

# ── Risk 1: Bus factor = 1 on critical path targets ───────────────────────────
risk_rows = []

for t in cp_set:
    bf = bus_factor_map.get(t, None)
    if bf is not None and bf <= 1:
        compile_ms = float(G.nodes[t].get('compile_time_sum_ms') or 0)
        risk_rows.append({
            'cmake_target': t,
            'risk_type': 'Bus factor = 1 on critical path',
            'risk_rating': 'High',
            'detail': f'bus_factor={bf}, compile_time={compile_ms:,.0f} ms',
            'compile_time_ms': compile_ms,
        })

# ── Risk 2: Core dependency depth ─────────────────────────────────────────────
from build_optimiser.graph import topological_depth
try:
    depth_map = topological_depth(G)
except Exception:
    depth_map = {}

for t in core_targets:
    depth = depth_map.get(t, 0)
    if depth >= 5:
        risk_rows.append({
            'cmake_target': t,
            'risk_type': 'Deep core dependency chain',
            'risk_rating': 'Medium' if depth < 8 else 'High',
            'detail': f'depth={depth} (core target deep in graph)',
            'compile_time_ms': float(G.nodes[t].get('compile_time_sum_ms') or 0),
        })

# ── Risk 3: Contested targets on critical path (bus factor analysis) ──────────
if not ownership_df.empty and 'is_contested' in ownership_df.columns:
    contested_cp = ownership_df[
        ownership_df['is_contested'] & ownership_df['cmake_target'].isin(cp_set)
    ]
    for _, row in contested_cp.iterrows():
        t = row['cmake_target']
        risk_rows.append({
            'cmake_target': t,
            'risk_type': 'Contested ownership on critical path',
            'risk_rating': 'Medium',
            'detail': f'hhi={row.get("ownership_hhi", "n/a"):.3f}, bus_factor={row.get("bus_factor", "n/a")}',
            'compile_time_ms': float(G.nodes[t].get('compile_time_sum_ms') or 0) if t in G else 0,
        })

# ── Risk 4: High codegen ratio + on critical path ─────────────────────────────
for _, row in target_df.iterrows():
    t = row['cmake_target']
    if t not in cp_set:
        continue
    ratio = float(row.get('codegen_ratio') or 0)
    if ratio >= 0.5:
        risk_rows.append({
            'cmake_target': t,
            'risk_type': 'High codegen ratio on critical path',
            'risk_rating': 'Medium',
            'detail': f'codegen_ratio={ratio:.2f}, files={int(row.get("codegen_file_count") or 0)}',
            'compile_time_ms': float(row.get('compile_time_sum_ms') or 0),
        })

# ── Build risk_df, guarding against empty ────────────────────────────────────
_RISK_COLS = ['cmake_target', 'risk_type', 'risk_rating', 'detail', 'compile_time_ms']
if risk_rows:
    risk_df = pd.DataFrame(risk_rows).sort_values(
        ['risk_rating', 'compile_time_ms'], ascending=[True, False]
    ).reset_index(drop=True)
else:
    risk_df = pd.DataFrame(columns=_RISK_COLS)

risk_colors = {'High': '#C44E52', 'Medium': 'darkorange', 'Low': 'steelblue'}

print(f'Risk items identified: {len(risk_df)}')
if risk_df.empty:
    print('(No risk items found — dataset may be too small for meaningful risk analysis.)')
print()
for rating in ['High', 'Medium', 'Low']:
    subset = risk_df[risk_df['risk_rating'] == rating]
    if subset.empty:
        continue
    print(f'=== {rating} Risk ({len(subset)} items) ===')
    for _, r in subset.iterrows():
        print(f'  [{r["risk_type"]}] {r["cmake_target"]}: {r["detail"]}')
    print()

# ── Risk summary chart ────────────────────────────────────────────────────────
if not risk_df.empty:
    fig, ax = plt.subplots(figsize=(10, max(4, len(risk_df) * 0.4)))
    colors = [risk_colors.get(r, 'grey') for r in risk_df['risk_rating']]
    ax.barh(range(len(risk_df)), risk_df['compile_time_ms'].values, color=colors, alpha=0.8)
    ax.set_yticks(range(len(risk_df)))
    ax.set_yticklabels(
        [f"[{r['risk_rating']}] {r['cmake_target'][:30]}" for _, r in risk_df.iterrows()],
        fontsize=8,
    )
    ax.invert_yaxis()
    ax.set_xlabel('Compile time (ms)')
    ax.set_title('Risk Items by Compile Time (coloured by severity)')
    ax.legend(
        handles=[Patch(color=c, label=r) for r, c in risk_colors.items()],
        loc='lower right', fontsize=8,
    )
    plt.tight_layout()
    plt.show()

# ── Save risk table for NB09 ──────────────────────────────────────────────────
results_dir = Path('../data/results')
results_dir.mkdir(parents=True, exist_ok=True)
risk_df.to_parquet('../data/results/risk_analysis.parquet', index=False)
print('Saved: data/results/risk_analysis.parquet')
